# Notebook 03 — Aplicaciones: POS Tagging con HMM

Aplicamos un HMM al **etiquetado morfosintáctico (POS tagging)**:
dada una oración, asignamos a cada palabra su categoría gramatical.

- **Estados ocultos:** DT (determinante), NN (sustantivo), VB (verbo), JJ (adjetivo)
- **Observaciones:** las palabras del texto
- **Algoritmo:** Viterbi en log-espacio

## 1. Vocabulario y etiquetas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

TAGS   = ['DT', 'NN', 'VB', 'JJ']
N_TAGS = len(TAGS)
tag_idx = {t: i for i, t in enumerate(TAGS)}

WORDS   = ['el', 'la', 'perro', 'gato', 'corre', 'salta', 'rapido', 'lento', 'un', 'una']
N_WORDS = len(WORDS)
word_idx = {w: i for i, w in enumerate(WORDS)}

print(f'Etiquetas ({N_TAGS}): {TAGS}')
print(f'Vocabulario ({N_WORDS}): {WORDS}')

## 2. Parámetros del HMM

Definidos manualmente para ilustrar la estructura.
En la práctica se aprenden con Baum-Welch de un corpus etiquetado.

In [ ]:
PI = np.array([0.50, 0.30, 0.10, 0.10])  # DT, NN, VB, JJ

# A[i][j] = P(etiqueta_j | etiqueta_i)
A = np.array([
#  DT    NN    VB    JJ
  [0.05, 0.80, 0.05, 0.10],  # DT ->
  [0.10, 0.10, 0.60, 0.20],  # NN ->
  [0.40, 0.25, 0.05, 0.30],  # VB ->
  [0.10, 0.60, 0.20, 0.10],  # JJ ->
])

# B[i][k] = P(palabra_k | etiqueta_i)
# Columnas: el, la, perro, gato, corre, salta, rapido, lento, un, una
B = np.array([
  [0.25, 0.25, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.25, 0.25],  # DT
  [0.00, 0.00, 0.50, 0.50, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],  # NN
  [0.00, 0.00, 0.00, 0.00, 0.50, 0.50, 0.00, 0.00, 0.00, 0.00],  # VB
  [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.50, 0.50, 0.00, 0.00],  # JJ
])

assert np.allclose(PI.sum(), 1.0)
assert np.allclose(A.sum(axis=1), 1.0)
assert np.allclose(B.sum(axis=1), 1.0)
print('Parametros validos. ✓')

## 3. Viterbi en log-espacio

In [ ]:
from scipy.special import logsumexp

def viterbi_log(obs_ids, pi, a, b):
    T_ = len(obs_ids)
    N_ = len(pi)
    log_pi = np.log(pi + 1e-300)
    log_a  = np.log(a  + 1e-300)
    log_b  = np.log(b  + 1e-300)

    delta = np.full((T_, N_), -np.inf)
    psi   = np.zeros((T_, N_), dtype=int)

    delta[0] = log_pi + log_b[:, obs_ids[0]]

    for t in range(1, T_):
        for j in range(N_):
            scores   = delta[t-1] + log_a[:, j]
            psi[t,j] = scores.argmax()
            delta[t,j] = scores[psi[t,j]] + log_b[j, obs_ids[t]]

    path = np.zeros(T_, dtype=int)
    path[-1] = delta[-1].argmax()
    for t in range(T_-2, -1, -1):
        path[t] = psi[t+1, path[t+1]]

    return path

print('Viterbi en log-espacio definido.')

## 4. Etiquetado de oraciones

In [ ]:
def tag(sentence):
    words = sentence.lower().split()
    ids = [word_idx.get(w, None) for w in words]
    if any(i is None for i in ids):
        unknown = [w for w, i in zip(words, ids) if i is None]
        print(f'  Palabras fuera del vocabulario: {unknown}')
        return None
    path = viterbi_log(ids, PI, A, B)
    result = list(zip(words, [TAGS[p] for p in path]))
    tagged = ' '.join(f'{w}/{t}' for w, t in result)
    print(f'  {sentence} -> {tagged}')
    return result

print('=== Etiquetado de oraciones ===')
oraciones = [
    'el perro corre',
    'un gato salta',
    'el perro corre rapido',
    'la gato salta lento',
]
resultados = [tag(o) for o in oraciones]
resultados = [r for r in resultados if r is not None]

## 5. Visualización de una oración etiquetada

In [ ]:
TAG_COLORS = {'DT': '#2E86AB', 'NN': '#27AE60', 'VB': '#E94F37', 'JJ': '#F39C12'}
TAG_DESC   = {'DT': 'Determinante', 'NN': 'Sustantivo', 'VB': 'Verbo', 'JJ': 'Adjetivo'}

demo_words = 'el perro corre rapido'.split()
demo_ids   = [word_idx[w] for w in demo_words]
demo_path  = viterbi_log(demo_ids, PI, A, B)
demo_tags  = [TAGS[p] for p in demo_path]

fig, ax = plt.subplots(figsize=(10, 3))
ax.set_xlim(-0.5, len(demo_words)-0.5)
ax.set_ylim(-0.6, 1.8)
ax.axis('off')
ax.set_title("POS Tagging con Viterbi — 'el perro corre rapido'",
             fontsize=12, fontweight='bold')

for i, (word, tag_lbl) in enumerate(zip(demo_words, demo_tags)):
    col = TAG_COLORS[tag_lbl]
    rect = mpatches.FancyBboxPatch((i-0.38, 0.55), 0.76, 0.70,
                                    boxstyle='round,pad=0.05',
                                    facecolor=col, alpha=0.85, edgecolor='white', lw=2)
    ax.add_patch(rect)
    ax.text(i, 0.90, tag_lbl, ha='center', va='center', fontsize=13,
            fontweight='bold', color='white')
    ax.text(i, 1.40, TAG_DESC[tag_lbl], ha='center', fontsize=8, color=col, style='italic')
    ax.text(i, 0.20, word, ha='center', va='center', fontsize=13, color='#2C3E50')
    if i < len(demo_words)-1:
        ax.annotate('', xy=(i+0.40, 0.90), xytext=(i+0.60, 0.90),
                   arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

patches = [mpatches.Patch(color=TAG_COLORS[t], label=f'{t} ({TAG_DESC[t]})') for t in TAGS]
ax.legend(handles=patches, loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

## 6. Discusión: HMM vs modelos modernos

| Aspecto | HMM | Transformer (BERT/GPT) |
|---------|:---:|:---------------------:|
| Contexto | Solo estado anterior | Toda la oración |
| Vocabulario fuera del dominio | Problema | Subword tokens (BPE) |
| Cantidad de datos necesarios | Poco (cientos de oraciones) | Mucho (millones) |
| Interpretabilidad | Alta (estados semánticos) | Baja |
| Velocidad de inferencia | Muy rápida (Viterbi lineal) | Más lenta |

**Cuándo usar un HMM:** cuando tienes pocos datos, necesitas interpretabilidad,
o el problema tiene estructura de estados discretos bien definida.

**Cuándo usar Transformers:** cuando tienes muchos datos y necesitas capturar
dependencias de largo alcance.